# Notebook 07 — Classification Metrics: Confusion Matrix, Precision, Recall & F1
## NordWind Energy Maintenance Ticket Classifier

**Course**: Analytics, Data Resources and Decision Making — Aarhus University  
**Lecture 6, Slot 5 Reinforcement** | 11 March 2026

---

### What this notebook does

In the slides you saw a worked example with 200 tickets. Here you'll run it **live**:

1. **50 synthetic maintenance tickets** from NordWind Energy (10 safety-critical, 40 routine)
2. **Gemini classifies** each ticket as `safety-critical` or `routine`
3. **You compute** the confusion matrix, accuracy, precision, recall, and F1
4. **(Optional)** Re-classify with a "cautious" prompt and see the precision–recall tradeoff in action
5. **Export** everything to Excel for further analysis

**Time to run**: ~3–5 minutes (API calls)


## Cell 1 — Setup & API Key

Install the Gemini SDK and enter your API key. Get a free key at [aistudio.google.com](https://aistudio.google.com/apikey).


In [ ]:
# Install the Gemini SDK
!pip install -q google-genai

import time
import getpass
import os

# Enter your Gemini API key (it won't be displayed)
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

from google import genai

client = genai.Client()
MODEL = "gemini-3.1-flash-lite-preview"

# Quick test
response = client.models.generate_content(
    model=MODEL,
    contents="Reply with only: OK"
)
print("API connection:", response.text.strip())


## Cell 2 — NordWind Maintenance Ticket Data

50 synthetic tickets: **10 safety-critical** and **40 routine** (a 20% base rate, matching the Slot 5 slides).

The interesting cases are tickets where **calm, professional language masks genuine urgency** — exactly the kind of edge case a classifier might get wrong.


In [ ]:
# ── 50 NordWind Energy maintenance tickets ──
# Sites: Ringkøbing, Thyborøn, Hvide Sande, Esbjerg Offshore, Hanstholm
# Turbines: NW-001 to NW-061 | Technicians: 8 names

tickets = [
    # ═══════════════════════════════════════════════════════════════
    # SAFETY-CRITICAL tickets (10)
    # ═══════════════════════════════════════════════════════════════

    # SC-1: Obvious — blade crack
    {
        "ticket_id": "TKT-201",
        "site": "Hvide Sande",
        "turbine_id": "NW-023",
        "technician": "Lars Eriksen",
        "text": "Visual inspection of NW-023 revealed a longitudinal crack approximately 40 cm along the trailing edge of blade 2, starting at the 18-metre mark. Crack appears to propagate from a prior lightning strike point. Turbine shut down immediately. Blade replacement required.",
        "ground_truth": "safety-critical"
    },
    # SC-2: Obvious — hydraulic failure
    {
        "ticket_id": "TKT-202",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-045",
        "technician": "Mette Hansen",
        "text": "Hydraulic pitch system on NW-045 failed during storm conditions. Blades unable to feather. Emergency shutdown triggered by overspeed protection at 22.4 RPM. Hydraulic fluid leak confirmed at cylinder B actuator seal. Immediate repair required before restart.",
        "ground_truth": "safety-critical"
    },
    # SC-3: Calm language masking urgency — bearing temperature
    {
        "ticket_id": "TKT-203",
        "site": "Ringkøbing",
        "turbine_id": "NW-012",
        "technician": "Anders Mortensen",
        "text": "Routine SCADA review for NW-012. Main bearing temperature reading at 94°C, which is 12°C above the seasonal baseline and 6°C below the automatic shutdown threshold. Rate of increase over the past 72 hours has been steady at approximately 2°C per day. Vibration levels within normal parameters. Grease condition appears adequate. Logging for continued monitoring.",
        "ground_truth": "safety-critical"
    },
    # SC-4: Calm language — pitch system anomaly
    {
        "ticket_id": "TKT-204",
        "site": "Thyborøn",
        "turbine_id": "NW-031",
        "technician": "Katrine Lund",
        "text": "Completed scheduled pitch system check on NW-031. Noted that blade 3 pitch angle deviates by 1.8° from the commanded position during load transitions. The deviation corrects itself within 4–6 seconds. System logs show this pattern has been present intermittently for approximately two weeks. No alarms triggered. Documenting for follow-up.",
        "ground_truth": "safety-critical"
    },
    # SC-5: Technical jargon obscuring severity — yaw misalignment
    {
        "ticket_id": "TKT-205",
        "site": "Hanstholm",
        "turbine_id": "NW-055",
        "technician": "Jens Petersen",
        "text": "NW-055 yaw drive motor current draw has increased to 47A during repositioning events (nominal: 28–32A). Yaw bearing backlash measured at 3.2 mm, exceeding the 2.0 mm service limit. Intermittent grinding noise reported during yaw operations in winds above 15 m/s. Yaw brake pad thickness at 4 mm (minimum: 6 mm). Unit remains operational.",
        "ground_truth": "safety-critical"
    },
    # SC-6: Calm language — tower bolt torque loss
    {
        "ticket_id": "TKT-206",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-048",
        "technician": "Lars Eriksen",
        "text": "Annual tower bolt inspection on NW-048. Section T3-T4 flange: 8 of 72 bolts measured below 70% of specified torque value. Three bolts showed visible corrosion on threads. Remaining sections within specification. Re-torquing completed on accessible bolts. Four bolts require specialized access equipment for re-torque. Scheduling follow-up.",
        "ground_truth": "safety-critical"
    },
    # SC-7: Obvious — fire detection
    {
        "ticket_id": "TKT-207",
        "site": "Ringkøbing",
        "turbine_id": "NW-008",
        "technician": "Mette Hansen",
        "text": "Fire detection system in NW-008 nacelle activated at 14:32. Automatic suppression system deployed. Turbine emergency stop executed. Source identified as electrical fault in the converter cabinet — charring visible on DC bus bar connections. No flame spread beyond the cabinet. Turbine offline pending full electrical inspection and suppression system recharge.",
        "ground_truth": "safety-critical"
    },
    # SC-8: Subtle — generator insulation degradation
    {
        "ticket_id": "TKT-208",
        "site": "Hvide Sande",
        "turbine_id": "NW-019",
        "technician": "Anders Mortensen",
        "text": "Generator insulation resistance test on NW-019 returned 1.8 MΩ at 1000V DC. Previous quarter reading was 3.2 MΩ; the quarter before that was 5.1 MΩ. Manufacturer minimum is 1.0 MΩ. Polarization index measured at 1.4 (acceptable range: >2.0). No visible damage to windings or connections. Ambient humidity was 78% at time of test. Results logged.",
        "ground_truth": "safety-critical"
    },
    # SC-9: Calm — gearbox oil analysis
    {
        "ticket_id": "TKT-209",
        "site": "Thyborøn",
        "turbine_id": "NW-037",
        "technician": "Katrine Lund",
        "text": "Quarterly gearbox oil analysis for NW-037 returned. Iron particle count at 142 ppm (previous: 68 ppm, baseline: 30–50 ppm). Copper at 28 ppm (previous: 12 ppm). Oil viscosity within spec. Gearbox operating normally per SCADA — no unusual vibration or temperature readings. Sample was drawn from the standard port following the usual procedure. Results forwarded to reliability engineering.",
        "ground_truth": "safety-critical"
    },
    # SC-10: Calm — CTV transfer safety issue
    {
        "ticket_id": "TKT-210",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-042",
        "technician": "Jens Petersen",
        "text": "During personnel transfer to NW-042, the CTV bow fender made contact with the transition piece access ladder, bending the lower handrail section approximately 15° inward. Transfer completed without injury. Handrail is now partially obstructing the step-off point. Sea state was 1.2 m Hs, within operational limits. Vessel captain noted unusual swell pattern from the northwest. Photographed and documented.",
        "ground_truth": "safety-critical"
    },

    # ═══════════════════════════════════════════════════════════════
    # ROUTINE tickets (40)
    # ═══════════════════════════════════════════════════════════════

    {
        "ticket_id": "TKT-101",
        "site": "Ringkøbing",
        "turbine_id": "NW-003",
        "technician": "Lars Eriksen",
        "text": "Completed 6-month scheduled lubrication on NW-003 main bearing, pitch bearings, and yaw gear. All grease levels topped up to specification. No unusual wear particles observed in old grease samples. Next service due August 2026.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-102",
        "site": "Thyborøn",
        "turbine_id": "NW-028",
        "technician": "Mette Hansen",
        "text": "Annual safety equipment inspection on NW-028. Climb assist system operational. Fall arrest anchors passed load test. First aid kit restocked. Emergency lighting batteries replaced. All items within certification period.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-103",
        "site": "Hvide Sande",
        "turbine_id": "NW-017",
        "technician": "Anders Mortensen",
        "text": "Replaced cabin air filters in NW-017 nacelle as part of scheduled maintenance. Old filters showed normal dust accumulation. HVAC system running at nominal airflow rates. Temperature control functioning correctly.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-104",
        "site": "Hanstholm",
        "turbine_id": "NW-052",
        "technician": "Katrine Lund",
        "text": "Quarterly SCADA data backup completed for NW-052. All historian data exported to central NordWind server. Communication link tested at 98.7% uptime over the quarter. Firmware version confirmed current.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-105",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-041",
        "technician": "Jens Petersen",
        "text": "Replaced aviation warning light on NW-041 nacelle roof. Previous light had reached end of rated lifespan (18,000 hours). New unit tested — flash pattern and intensity confirmed compliant with Danish Civil Aviation Authority requirements.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-106",
        "site": "Ringkøbing",
        "turbine_id": "NW-005",
        "technician": "Frederik Nielsen",
        "text": "Routine visual inspection of NW-005 tower exterior. Paint condition good, no corrosion spots detected. Foundation grout intact. Drainage channels clear. Bird nesting material removed from transformer housing ventilation grille.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-107",
        "site": "Thyborøn",
        "turbine_id": "NW-033",
        "technician": "Sofie Andersen",
        "text": "Updated NW-033 controller software to version 4.2.1 per manufacturer advisory. No operational changes — patch addresses a minor logging timestamp format issue. Turbine restarted successfully after update. Production resumed at 15:22.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-108",
        "site": "Hvide Sande",
        "turbine_id": "NW-021",
        "technician": "Lars Eriksen",
        "text": "Calibrated wind speed and direction sensors on NW-021 nacelle anemometer. Readings compared against portable reference unit — deviation within ±0.3 m/s and ±2°. No adjustment needed. Calibration certificate updated.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-109",
        "site": "Hanstholm",
        "turbine_id": "NW-058",
        "technician": "Mette Hansen",
        "text": "Conducted annual electrical thermography scan on NW-058 main switchgear and converter. All connections within normal temperature range. No hot spots detected. Report filed with electrical safety documentation.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-110",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-046",
        "technician": "Anders Mortensen",
        "text": "Replaced worn CTV mooring line on vessel Havørn. Old line showed normal surface wear consistent with usage hours. New line rated to 12-tonne breaking load. Splicing and termination inspected by vessel captain. Line register updated.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-111",
        "site": "Ringkøbing",
        "turbine_id": "NW-009",
        "technician": "Katrine Lund",
        "text": "Cleaned NW-009 blade leading edges as part of annual maintenance. Light erosion visible on blade tips — within acceptable limits per manufacturer guidelines. No protective tape replacement needed at this time. Drone photos archived.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-112",
        "site": "Thyborøn",
        "turbine_id": "NW-029",
        "technician": "Jens Petersen",
        "text": "Replaced hydraulic filter on NW-029 pitch system. Old filter showed normal discoloration. Hydraulic fluid level topped up with 2.5 litres. System pressure tested at 245 bar — within specification (240–260 bar).",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-113",
        "site": "Hvide Sande",
        "turbine_id": "NW-024",
        "technician": "Frederik Nielsen",
        "text": "Inspected NW-024 generator brush gear during scheduled downtime. Brush wear at approximately 40% — replacement not yet required. Slip ring surface in good condition, no scoring visible. Dust extraction system functioning normally.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-114",
        "site": "Hanstholm",
        "turbine_id": "NW-054",
        "technician": "Sofie Andersen",
        "text": "Tightened loose cable tray bracket in NW-054 tower section 2. Bracket had vibrated partially free of its mounting — one of four bolts had backed out. Re-secured with thread-locking compound. No cable damage observed.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-115",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-043",
        "technician": "Lars Eriksen",
        "text": "Monthly inspection of NW-043 corrosion protection system. Sacrificial anodes on transition piece measured — remaining thickness 62% of original. Impressed current system output confirmed at 2.4A. Both systems operating within design parameters.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-116",
        "site": "Ringkøbing",
        "turbine_id": "NW-001",
        "technician": "Mette Hansen",
        "text": "Replaced door seal on NW-001 tower base access door. Old seal was brittle and allowing minor water ingress during heavy rain. New EPDM seal fitted and tested with water spray. Interior dry after test.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-117",
        "site": "Thyborøn",
        "turbine_id": "NW-035",
        "technician": "Anders Mortensen",
        "text": "Tested UPS battery bank in NW-035 base. All 24 cells within voltage specification (12.4–12.8V per cell). Load test sustained 15-minute backup at rated capacity. Batteries are 2 years into 5-year expected lifespan.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-118",
        "site": "Hvide Sande",
        "turbine_id": "NW-020",
        "technician": "Katrine Lund",
        "text": "Greased NW-020 yaw ring gear teeth. Applied Mobilith SHC 460 per manufacturer specification. Yaw motor operated through full 360° rotation during greasing to ensure coverage. Operation smooth, no unusual noise.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-119",
        "site": "Hanstholm",
        "turbine_id": "NW-056",
        "technician": "Jens Petersen",
        "text": "Replaced expired fire extinguishers in NW-056 nacelle and tower base. Two CO2 units and one dry powder unit exchanged. Old units returned for servicing. Locations marked on updated safety plan.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-120",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-047",
        "technician": "Frederik Nielsen",
        "text": "Performed bi-annual bolt torque check on NW-047 blade hub. All 216 bolts within specification (±5% of target torque). No bolts required re-torquing. Next check scheduled for September 2026.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-121",
        "site": "Ringkøbing",
        "turbine_id": "NW-010",
        "technician": "Mikkel Sørensen",
        "text": "Cleared vegetation growing at the base of NW-010 foundation. Grass and small shrubs were encroaching on the access path and blocking the drainage outlet. Area cleared and gravel path restored.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-122",
        "site": "Thyborøn",
        "turbine_id": "NW-030",
        "technician": "Lars Eriksen",
        "text": "Replaced cabin heater element in NW-030 nacelle. Old element had failed — nacelle interior temperature was 8°C during last visit (target: 15–20°C for electronics). New heater tested at full output. Thermostat set to 18°C.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-123",
        "site": "Hvide Sande",
        "turbine_id": "NW-022",
        "technician": "Mette Hansen",
        "text": "Inspected NW-022 lightning protection system. Down conductor resistance measured at 0.8 Ω (specification: <1.0 Ω). Blade receptor tips intact. Earth pit connection verified. System compliant for the current certification period.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-124",
        "site": "Hanstholm",
        "turbine_id": "NW-059",
        "technician": "Anders Mortensen",
        "text": "Monthly check of NW-059 condition monitoring system. All vibration sensors reporting. Accelerometer on main bearing replaced due to intermittent signal dropout — faulty connector identified and replaced. System now fully operational.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-125",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-044",
        "technician": "Katrine Lund",
        "text": "Repainted touch-up areas on NW-044 transition piece where coating had been damaged during recent maintenance access. Marine-grade epoxy primer and topcoat applied. Cure time 48 hours — area flagged for no-contact period.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-126",
        "site": "Ringkøbing",
        "turbine_id": "NW-007",
        "technician": "Jens Petersen",
        "text": "Annual oil sample from NW-007 gearbox collected and sent to laboratory. Visual check of oil colour and clarity — normal. Oil level at sight glass midpoint. No leaks detected at any seal or drain point.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-127",
        "site": "Thyborøn",
        "turbine_id": "NW-032",
        "technician": "Frederik Nielsen",
        "text": "Replaced worn brake pads on NW-032 rotor lock. Old pads measured at 8 mm (minimum: 6 mm) — replaced proactively during scheduled downtime. Brake function tested: rotor held at rated torque for 30 seconds. Satisfactory.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-128",
        "site": "Hvide Sande",
        "turbine_id": "NW-018",
        "technician": "Sofie Andersen",
        "text": "Cleaned optical fibre connectors on NW-018 SCADA communication trunk. Signal attenuation had increased slightly over the past month — attributed to dust ingress. After cleaning, signal levels returned to -14 dBm (nominal: -12 to -16 dBm).",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-129",
        "site": "Hanstholm",
        "turbine_id": "NW-053",
        "technician": "Lars Eriksen",
        "text": "Installed new weather station on NW-053 nacelle as part of fleet-wide met data upgrade. Sensors include anemometer, wind vane, temperature, humidity, and barometric pressure. Commissioning complete — data streaming to central SCADA.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-130",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-049",
        "technician": "Mette Hansen",
        "text": "Performed annual crane inspection on NW-049 nacelle service crane. Wire rope within wear limits. Hoist brake test passed. Load cell calibrated. Safety latch on hook confirmed operational. Crane logbook updated.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-131",
        "site": "Ringkøbing",
        "turbine_id": "NW-004",
        "technician": "Anders Mortensen",
        "text": "Tightened loose bolt on NW-004 nacelle cover panel. Panel was rattling audibly in winds above 12 m/s. Bolt had vibrated loose — re-secured with Nordlock washer. No structural concern.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-132",
        "site": "Thyborøn",
        "turbine_id": "NW-036",
        "technician": "Katrine Lund",
        "text": "Replaced NW-036 transformer cooling fan. Fan motor had reached 42,000 operating hours — manufacturer recommends replacement at 40,000 hours. New fan tested at full speed. Transformer operating temperature stable at 58°C.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-133",
        "site": "Hvide Sande",
        "turbine_id": "NW-016",
        "technician": "Jens Petersen",
        "text": "Checked and re-tensioned drive belt on NW-016 generator cooling system. Belt showed minor fraying on one edge but tension was within specification after adjustment. Scheduled full belt replacement for next planned downtime.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-134",
        "site": "Hanstholm",
        "turbine_id": "NW-057",
        "technician": "Frederik Nielsen",
        "text": "Removed bird droppings accumulation from NW-057 nacelle roof around anemometer mast. Accumulation was not affecting sensor readings but was causing minor corrosion on the mast base plate. Cleaned and applied protective wax coating.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-135",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-050",
        "technician": "Sofie Andersen",
        "text": "Tested emergency lighting system in NW-050 tower and nacelle. All 14 emergency light units functional. Battery backup duration confirmed at >90 minutes (requirement: 60 minutes). Two bulbs replaced proactively due to slight dimming.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-136",
        "site": "Ringkøbing",
        "turbine_id": "NW-006",
        "technician": "Lars Eriksen",
        "text": "Documented NW-006 power curve data for monthly performance report. Energy yield tracking 3% above P50 estimate for this period. Availability at 97.2%. No anomalies noted in operational data.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-137",
        "site": "Thyborøn",
        "turbine_id": "NW-034",
        "technician": "Mette Hansen",
        "text": "Serviced NW-034 pitch accumulator. Nitrogen pre-charge checked at 105 bar (specification: 100–110 bar). Bladder integrity confirmed via pressure hold test. Hydraulic fluid topped up by 0.3 litres to compensate for minor seepage at fitting.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-138",
        "site": "Hvide Sande",
        "turbine_id": "NW-025",
        "technician": "Anders Mortensen",
        "text": "Replaced faded and peeling turbine identification decals on NW-025 tower base. Old decals no longer legible at safe distance. New reflective decals applied per NordWind visual identity standards.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-139",
        "site": "Hanstholm",
        "turbine_id": "NW-060",
        "technician": "Katrine Lund",
        "text": "Conducted bi-annual elevator inspection on NW-060 service lift. All safety interlocks tested and functional. Cable wear within limits. Emergency descent procedure verified. Inspection certificate renewed until March 2027.",
        "ground_truth": "routine"
    },
    {
        "ticket_id": "TKT-140",
        "site": "Esbjerg Offshore",
        "turbine_id": "NW-051",
        "technician": "Jens Petersen",
        "text": "Performed planned J-tube inspection on NW-051 using ROV survey. Cable entry seals intact. No marine growth obstructing the cable path. Scour protection at foundation base shows minor settlement — within design tolerance. Video records archived.",
        "ground_truth": "routine"
    },
]

print(f"Total tickets: {len(tickets)}")
print(f"Safety-critical: {sum(1 for t in tickets if t['ground_truth'] == 'safety-critical')}")
print(f"Routine: {sum(1 for t in tickets if t['ground_truth'] == 'routine')}")


## Cell 3 — Classify Each Ticket with Gemini

We send each ticket to Gemini with a simple classification prompt. The model must respond with **only** the label: `safety-critical` or `routine`.

This takes a few minutes — one API call per ticket with a short delay to avoid rate limiting.


In [ ]:
import time

CLASSIFICATION_PROMPT = """You are a maintenance ticket classifier for NordWind Energy, a wind energy company.

Read the following maintenance ticket and classify it as either "safety-critical" or "routine".

A ticket is "safety-critical" if it describes any condition that could lead to equipment damage, environmental harm, or risk to personnel if not addressed promptly. This includes structural failures, safety system activations, degrading conditions approaching alarm thresholds, and access/transfer safety incidents.

A ticket is "routine" if it describes scheduled maintenance, minor repairs, inspections with satisfactory results, or administrative tasks.

Respond with ONLY the label — either: safety-critical
or: routine

Nothing else.

TICKET:
{ticket_text}"""

predictions_default = []

for i, ticket in enumerate(tickets):
    prompt = CLASSIFICATION_PROMPT.format(ticket_text=ticket["text"])
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )
        pred = response.text.strip().lower()
        # Normalise: accept minor variations
        if "safety" in pred or "critical" in pred:
            pred = "safety-critical"
        elif "routine" in pred:
            pred = "routine"
        else:
            pred = response.text.strip()  # keep raw if unexpected
    except Exception as e:
        pred = f"ERROR: {e}"
    
    predictions_default.append(pred)
    status = "✓" if pred == ticket["ground_truth"] else "✗"
    print(f"  [{i+1:2d}/50] {ticket['ticket_id']} | True: {ticket['ground_truth']:16s} | Pred: {pred:16s} {status}")
    time.sleep(1.5)  # rate limit buffer

# Count valid predictions
valid = [p for p in predictions_default if p in ("safety-critical", "routine")]
print(f"\nClassification complete. {len(valid)}/50 valid predictions.")
if len(valid) < 50:
    invalid = [(i, p) for i, p in enumerate(predictions_default) if p not in ("safety-critical", "routine")]
    print(f"Invalid responses: {invalid}")


## Cell 4 — Build the Confusion Matrix

We compare predictions to ground truth and compute TP, FP, TN, FN.

**Positive class** = `safety-critical`  
**Negative class** = `routine`


In [ ]:
import pandas as pd

# Compute confusion matrix values
ground_truths = [t["ground_truth"] for t in tickets]

TP = sum(1 for gt, pred in zip(ground_truths, predictions_default) if gt == "safety-critical" and pred == "safety-critical")
FN = sum(1 for gt, pred in zip(ground_truths, predictions_default) if gt == "safety-critical" and pred == "routine")
FP = sum(1 for gt, pred in zip(ground_truths, predictions_default) if gt == "routine" and pred == "safety-critical")
TN = sum(1 for gt, pred in zip(ground_truths, predictions_default) if gt == "routine" and pred == "routine")

# Display as a clean DataFrame
confusion_df = pd.DataFrame(
    [[TP, FN], [FP, TN]],
    index=["Actual: Safety-Critical", "Actual: Routine"],
    columns=["Predicted: Safety-Critical", "Predicted: Routine"]
)

print("═" * 60)
print("  CONFUSION MATRIX — Default Prompt")
print("═" * 60)
print()
print(confusion_df.to_string())
print()
print(f"  Row totals:  Critical = {TP + FN},  Routine = {FP + TN}")
print(f"  Col totals:  Pred Critical = {TP + FP},  Pred Routine = {FN + TN}")
print(f"  Total: {TP + FN + FP + TN}")


## Cell 5 — Compute Accuracy, Precision, Recall, F1

These four metrics tell different parts of the story. Pay attention to how they diverge — that's the lesson.


In [ ]:
# ── Compute metrics ──

total = TP + TN + FP + FN

accuracy = (TP + TN) / total if total > 0 else 0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("═" * 60)
print("  CLASSIFICATION METRICS — Default Prompt")
print("═" * 60)
print()
print(f"  Accuracy  = (TP + TN) / Total = ({TP} + {TN}) / {total} = {accuracy:.1%}")
print(f"    → The classifier is correct {accuracy:.1%} of the time.")
print()
print(f"  Precision = TP / (TP + FP)     = {TP} / ({TP} + {FP}) = {precision:.1%}")
print(f"    → {precision:.0%} of tickets flagged as critical were actually critical.")
print()
print(f"  Recall    = TP / (TP + FN)     = {TP} / ({TP} + {FN}) = {recall:.1%}")
print(f"    → We caught {recall:.0%} of truly critical tickets.")
if FN > 0:
    print(f"    ⚠ {FN} safety-critical ticket(s) were classified as routine!")
print()
print(f"  F1 Score  = 2 × (P × R)/(P + R) = {f1:.2f}")
print(f"    → The harmonic mean balances precision and recall.")
print()
print("─" * 60)

if accuracy > 0.85 and recall < 0.90:
    print("  ⚠ NOTE: High accuracy can be misleading when the base rate")
    print("    is low (only 20% critical). Check precision and recall!")


## Cell 6 — (Optional) The Cautious Prompt: Precision–Recall Tradeoff

What if we tell the model to **err on the side of caution**? This should increase recall (catch more critical tickets) but may decrease precision (more false alarms).

This is the precision–recall tradeoff from the slides, demonstrated live.


In [ ]:
CAUTIOUS_PROMPT = """You are a maintenance ticket classifier for NordWind Energy, a wind energy company.

Read the following maintenance ticket and classify it as either "safety-critical" or "routine".

IMPORTANT: Err on the side of caution. If there is ANY possibility that a ticket might involve a safety concern — even if the language is calm or the issue seems minor — classify it as "safety-critical". It is better to flag something unnecessarily than to miss a genuine safety issue.

Respond with ONLY the label — either: safety-critical
or: routine

Nothing else.

TICKET:
{ticket_text}"""

predictions_cautious = []

print("Running cautious prompt classification...")
print()

for i, ticket in enumerate(tickets):
    prompt = CAUTIOUS_PROMPT.format(ticket_text=ticket["text"])
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )
        pred = response.text.strip().lower()
        if "safety" in pred or "critical" in pred:
            pred = "safety-critical"
        elif "routine" in pred:
            pred = "routine"
        else:
            pred = response.text.strip()
    except Exception as e:
        pred = f"ERROR: {e}"
    
    predictions_cautious.append(pred)
    status = "✓" if pred == ticket["ground_truth"] else "✗"
    print(f"  [{i+1:2d}/50] {ticket['ticket_id']} | True: {ticket['ground_truth']:16s} | Pred: {pred:16s} {status}")
    time.sleep(1.5)

# ── Compute cautious metrics ──
TP_c = sum(1 for gt, p in zip(ground_truths, predictions_cautious) if gt == "safety-critical" and p == "safety-critical")
FN_c = sum(1 for gt, p in zip(ground_truths, predictions_cautious) if gt == "safety-critical" and p == "routine")
FP_c = sum(1 for gt, p in zip(ground_truths, predictions_cautious) if gt == "routine" and p == "safety-critical")
TN_c = sum(1 for gt, p in zip(ground_truths, predictions_cautious) if gt == "routine" and p == "routine")

total_c = TP_c + TN_c + FP_c + FN_c
accuracy_c = (TP_c + TN_c) / total_c if total_c > 0 else 0
precision_c = TP_c / (TP_c + FP_c) if (TP_c + FP_c) > 0 else 0
recall_c = TP_c / (TP_c + FN_c) if (TP_c + FN_c) > 0 else 0
f1_c = 2 * (precision_c * recall_c) / (precision_c + recall_c) if (precision_c + recall_c) > 0 else 0

# ── Side-by-side comparison ──
print()
print("═" * 65)
print("  COMPARISON: Default Prompt vs. Cautious Prompt")
print("═" * 65)
print()

comparison_df = pd.DataFrame({
    "Default Prompt": [
        f"{TP}", f"{FP}", f"{FN}", f"{TN}",
        f"{accuracy:.1%}", f"{precision:.1%}", f"{recall:.1%}", f"{f1:.2f}"
    ],
    "Cautious Prompt": [
        f"{TP_c}", f"{FP_c}", f"{FN_c}", f"{TN_c}",
        f"{accuracy_c:.1%}", f"{precision_c:.1%}", f"{recall_c:.1%}", f"{f1_c:.2f}"
    ]
}, index=[
    "True Positives (correctly flagged critical)",
    "False Positives (false alarms)",
    "False Negatives (missed critical ⚠)",
    "True Negatives (correctly cleared)",
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
])

print(comparison_df.to_string())
print()
print("─" * 65)
print("  INTERPRETATION:")
print()
if recall_c > recall:
    print(f"  ↑ Recall increased: {recall:.1%} → {recall_c:.1%}")
    print(f"    The cautious prompt catches more truly critical tickets.")
if precision_c < precision:
    print(f"  ↓ Precision decreased: {precision:.1%} → {precision_c:.1%}")
    print(f"    But it also creates more false alarms for the ops team.")
if FN_c < FN:
    print(f"  ↓ Missed critical tickets: {FN} → {FN_c}")
elif FN_c == FN:
    print(f"  = Missed critical tickets unchanged: {FN}")
print()
print("  This is the precision–recall tradeoff: catching more true")
print("  positives comes at the cost of more false positives.")
print("  Which balance is right depends on the business context.")


## Cell 7 — Export Results to Excel

Download the full results — every ticket with both predictions — plus a summary sheet with confusion matrices and metrics.


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

# ── Sheet 1: Ticket-level results ──
ws1 = wb.active
ws1.title = "Ticket Results"

# Header style
header_font = Font(bold=True, color="FFFFFF", size=11)
header_fill = PatternFill(start_color="1B3A5C", end_color="1B3A5C", fill_type="solid")
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

headers = [
    "Ticket ID", "Site", "Turbine ID", "Technician", "Ticket Text",
    "Ground Truth", "Prediction (Default)", "Correct? (Default)",
    "Prediction (Cautious)", "Correct? (Cautious)"
]

for col, header in enumerate(headers, 1):
    cell = ws1.cell(row=1, column=col, value=header)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal='center', wrap_text=True)
    cell.border = thin_border

# Data rows
green_fill = PatternFill(start_color="E2EFDA", end_color="E2EFDA", fill_type="solid")
red_fill = PatternFill(start_color="FCE4EC", end_color="FCE4EC", fill_type="solid")

for i, ticket in enumerate(tickets):
    row = i + 2
    correct_default = predictions_default[i] == ticket["ground_truth"]
    correct_cautious = predictions_cautious[i] == ticket["ground_truth"]
    
    values = [
        ticket["ticket_id"], ticket["site"], ticket["turbine_id"],
        ticket["technician"], ticket["text"], ticket["ground_truth"],
        predictions_default[i], "✓" if correct_default else "✗",
        predictions_cautious[i], "✓" if correct_cautious else "✗"
    ]
    for col, val in enumerate(values, 1):
        cell = ws1.cell(row=row, column=col, value=val)
        cell.border = thin_border
        cell.alignment = Alignment(wrap_text=(col == 5), vertical='top')
        # Highlight incorrect predictions
        if col == 8:
            cell.fill = green_fill if correct_default else red_fill
        if col == 10:
            cell.fill = green_fill if correct_cautious else red_fill

# Column widths
widths = [10, 16, 10, 16, 60, 16, 18, 10, 18, 10]
for i, w in enumerate(widths, 1):
    ws1.column_dimensions[get_column_letter(i)].width = w

# ── Sheet 2: Summary metrics ──
ws2 = wb.create_sheet("Summary Metrics")

def write_section(ws, start_row, title, tp, fp, fn, tn, acc, prec, rec, f1_val):
    ws.cell(row=start_row, column=1, value=title).font = Font(bold=True, size=13)
    
    # Confusion matrix
    r = start_row + 2
    ws.cell(row=r, column=2, value="Predicted: Critical").font = Font(bold=True)
    ws.cell(row=r, column=3, value="Predicted: Routine").font = Font(bold=True)
    ws.cell(row=r+1, column=1, value="Actual: Critical").font = Font(bold=True)
    ws.cell(row=r+1, column=2, value=tp)
    ws.cell(row=r+1, column=3, value=fn)
    ws.cell(row=r+2, column=1, value="Actual: Routine").font = Font(bold=True)
    ws.cell(row=r+2, column=2, value=fp)
    ws.cell(row=r+2, column=3, value=tn)
    
    for row in range(r, r+3):
        for col in range(1, 4):
            ws.cell(row=row, column=col).border = thin_border
    
    # Metrics
    r2 = r + 4
    metrics = [
        ("Accuracy", f"{acc:.1%}"),
        ("Precision", f"{prec:.1%}"),
        ("Recall", f"{rec:.1%}"),
        ("F1 Score", f"{f1_val:.2f}"),
    ]
    for j, (name, val) in enumerate(metrics):
        ws.cell(row=r2+j, column=1, value=name).font = Font(bold=True)
        ws.cell(row=r2+j, column=2, value=val)
    
    return r2 + len(metrics) + 2

ws2.column_dimensions['A'].width = 22
ws2.column_dimensions['B'].width = 22
ws2.column_dimensions['C'].width = 22

next_row = write_section(ws2, 1, "Default Prompt", TP, FP, FN, TN, accuracy, precision, recall, f1)
write_section(ws2, next_row, "Cautious Prompt", TP_c, FP_c, FN_c, TN_c, accuracy_c, precision_c, recall_c, f1_c)

# Save
output_path = "07_classification_metrics_results.xlsx"
wb.save(output_path)

# Download link for Colab
try:
    from google.colab import files
    files.download(output_path)
    print(f"✅ Downloading {output_path}...")
except ImportError:
    print(f"✅ Results saved to {output_path}")
    print("   (Run in Google Colab for automatic download)")


---

## What to take away

1. **Accuracy can be misleading** when the base rate is imbalanced (only 20% of tickets are critical). A classifier that labels everything "routine" would already be 80% accurate.

2. **Precision and recall tell different stories**: precision measures false alarm rate; recall measures the miss rate on critical tickets. Which matters more depends on the business context.

3. **The prompt matters**: a more cautious prompt shifts the tradeoff — fewer missed critical tickets, but more false alarms. This is a business decision, not just a technical one.

4. **The confusion matrix is your friend**: always look at the actual numbers before trusting a headline metric.

---

*Next up in Day 2: Slot 6 covers error analysis — looking at actual AI outputs to discover failure patterns that metrics alone can't reveal.*
